[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/12_NLP/45_sentiment_analysis.ipynb)

> 📎 **Optional module — reference style.** Module 12 is optional. Like the appendices, these notebooks are written as a demo / reference: they focus on *seeing* a library at work rather than interactive exercises. Each notebook is built to run end-to-end *without* the optional library — it falls back to a small built-in stand-in — so you can read and run it offline. Install the optional library (see **Install** below) to swap the stand-in for the real thing.

---

# 📓 Notebook 45 — Sentiment Analysis

> **Module:** 12 · Optional · **Type:** Reference · **Estimated time:** 45–75 min · **Difficulty:** Beginner → Intermediate

**Sentiment analysis** turns free text — product reviews, support tickets, NPS verbatims, app-store comments, social mentions — into a *signal you can aggregate*: positive / negative / neutral, or a numeric polarity score. That signal is what lets a business answer questions like *"did sentiment drop after the price change?"*, *"which feature do customers complain about most?"*, or *"route this angry ticket to a human now."*

There is no single "sentiment model." There is a **ladder** of techniques, each trading effort for power. This notebook climbs all three rungs, and — importantly — shows *when each one is the right tool*, because reaching for a transformer when a 20-line rule would do is a classic junior mistake.

1. **Lexicon / rule-based** (VADER) — no training, instant, transparent. Great for a quick read on short social-style text.
2. **Classical ML** (TF-IDF + Logistic Regression) — the *workhorse baseline*. Cheap, fast, interpretable, and shockingly hard to beat on in-domain data. Build this one well and you will use it for years.
3. **Transformers** (Hugging Face) — pretrained deep models that understand context and negation far better, at the cost of size and latency.

## 🎯 Learning objectives

By the end you will be able to:

- Explain the **three families** of sentiment methods and pick one for a given business constraint (latency, labels, accuracy, interpretability).
- Run a **lexicon scorer** and read its `compound` score, and explain the linguistic tricks (negation, intensifiers, emoji, CAPS, punctuation) VADER handles that a naive word list does not.
- Train, evaluate, and **inspect** a TF-IDF + LogisticRegression classifier — including reading `.coef_` to see *which words drive the prediction*.
- Call a Hugging Face `pipeline("sentiment-analysis")` and name a few domain-specific models (FinBERT, twitter-roberta).
- Describe **aspect-based** sentiment and the **pitfalls** (sarcasm, domain shift, neutral class, class imbalance, calibration, stars→sentiment).

## ✅ Prerequisites

- **NB 14 — sklearn basics** (train/test split, `fit`/`predict`, metrics). This is the main dependency.
- **NB 43 / 44 — topic modeling** *(optional)* — helpful background on turning text into features, but not required.

## 📦 Install

Everything below runs offline with a built-in stand-in. To use the *real* libraries:

```bash
pip install vaderSentiment transformers torch
```

All three are **optional**. `vaderSentiment` is tiny; `transformers torch` are large (hundreds of MB) and will download a model on first use.

## 0. Setup & imports

We import numpy / matplotlib / seaborn for plotting and sklearn for the classical model. The two optional libraries are imported behind **capability flags** (`HAS_VADER`, `HAS_TRANSFORMERS`) so a missing install never crashes the notebook — a pattern you have seen throughout Module 12.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# --- optional library 1: VADER (lexicon) ---
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    HAS_VADER = True
except Exception:
    HAS_VADER = False

# --- optional library 2: Hugging Face transformers ---
try:
    from transformers import pipeline
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False

print(f"VADER installed:        {HAS_VADER}")
print(f"transformers installed: {HAS_TRANSFORMERS}")
print("\nEither way, this notebook runs end-to-end. "
      "Flags only decide real-vs-stand-in.")

### A handful of example texts

We will reuse this small set of business-flavoured snippets across all three methods, so you can compare how each one scores the *same* sentences — including the deliberately tricky ones (negation, sarcasm, mixed).

In [ ]:
examples = [
    "I absolutely love this laptop, best purchase of the year!",
    "The battery life is terrible and it overheats constantly.",
    "Delivery was fine, nothing special.",
    "This is NOT a good product. Avoid it.",            # negation
    "Great. Another broken update. Just what I needed.", # sarcasm
    "The screen is gorgeous but the price is outrageous.", # mixed
]
for t in examples:
    print("•", t)

## 1. Lexicon / rule-based sentiment — VADER

**VADER** (*Valence Aware Dictionary and sEntiment Reasoner*) is a **rule-based** model. There is no training step. It ships with a hand-curated dictionary that maps ~7,500 words/emoji to a *valence* score (how positive or negative they are), plus a set of grammatical rules. You feed it text, it hands back four numbers — instantly.

### Why it punches above its weight

A naive approach — "count positive words minus negative words" — fails on real text. VADER was specifically engineered to handle the things that flip or amplify sentiment in **short, informal** text (tweets, reviews, chat):

| Phenomenon | Example | What VADER does |
|---|---|---|
| **Negation** | *not good* | flips the valence of the following words |
| **Intensifiers** | *very good*, *extremely bad* | boosts magnitude |
| **ALL-CAPS** | *GREAT* vs *great* | increases intensity |
| **Punctuation** | *good!!!* | exclamation marks raise intensity |
| **Emoji / emoticons** | 😍, `:)` | mapped to valence directly |
| **Contrastive *but*** | *good but slow* | weights the clause after *but* |

### The API

```python
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
analyzer.polarity_scores("I love it!")
# -> {'neg': 0.0, 'neu': 0.295, 'pos': 0.705, 'compound': 0.6696}
```

`neg / neu / pos` are *proportions* that sum to 1. The one you usually act on is **`compound`**: a single score in **[-1, +1]**. The standard thresholds are:

- `compound >= 0.05` → **positive**
- `compound <= -0.05` → **negative**
- otherwise → **neutral**

### Offline stand-in

If VADER isn't installed we drop in a *tiny* pure-Python scorer. It is deliberately simple — a small word list plus one negation rule and a CAPS boost — so you can read exactly how a lexicon scorer works. It is **not** as good as real VADER (smaller dictionary, fewer rules), which is itself a useful lesson: the value of VADER is in the *curation and rules*, not the idea.

In [ ]:
# A minimal lexicon scorer used ONLY when real VADER is unavailable.
import re

_POS = {
    "love": 2.5, "loved": 2.5, "great": 2.0, "good": 1.5, "gorgeous": 2.0,
    "best": 2.5, "excellent": 2.5, "amazing": 2.5, "fine": 0.8, "happy": 1.8,
    "fast": 1.0, "recommend": 1.5, "helpful": 1.8,
}
_NEG = {
    "terrible": -2.5, "bad": -1.5, "broken": -2.0, "overheats": -1.5,
    "avoid": -1.8, "outrageous": -1.5, "slow": -1.2, "hate": -2.5,
    "worst": -2.5, "awful": -2.5, "disappointed": -1.8, "disaster": -2.5,
}
# Words that flip the sentiment of the next valence word. Note 'avoid' is a
# *negative* word (see _NEG), not a negator -- "avoid it" is itself negative.
_NEGATIONS = {"not", "no", "n't", "never"}
_INTENSIFIERS = {"very": 1.5, "absolutely": 1.7, "extremely": 1.7,
                 "really": 1.4, "so": 1.3}

def _standin_scores(text):
    tokens = re.findall(r"[A-Za-z']+|[!]", text)
    total = 0.0
    boost = 1.0
    negate = False
    for tok in tokens:
        low = tok.lower()
        is_caps = tok.isupper() and len(tok) > 1
        if low in _INTENSIFIERS:
            boost *= _INTENSIFIERS[low]
            continue
        if low in _NEGATIONS:
            negate = True
            continue
        val = _POS.get(low, 0.0) + _NEG.get(low, 0.0)
        if tok == "!":
            total *= 1.15  # punctuation raises intensity (toy version)
            continue
        if val != 0.0:
            if is_caps:
                val *= 1.3
            val *= boost
            if negate:
                val *= -0.8
            total += val
            boost, negate = 1.0, False
    # squash to [-1, 1] like VADER's compound (alpha-normalisation)
    compound = total / np.sqrt(total ** 2 + 15)
    pos = max(compound, 0); neg = max(-compound, 0)
    neu = max(0.0, 1 - pos - neg)
    s = pos + neg + neu or 1.0
    return {"neg": round(neg/s, 3), "neu": round(neu/s, 3),
            "pos": round(pos/s, 3), "compound": round(compound, 4)}

# Unified getter: real VADER if available, else the stand-in.
if HAS_VADER:
    _analyzer = SentimentIntensityAnalyzer()
    vader_scores = _analyzer.polarity_scores
    print("Using REAL VADER.")
else:
    vader_scores = _standin_scores
    print("Using built-in STAND-IN lexicon scorer (install vaderSentiment "
          "for the real thing).")

In [ ]:
def vader_label(text, pos_t=0.05, neg_t=-0.05):
    c = vader_scores(text)["compound"]
    if c >= pos_t:
        return "positive"
    if c <= neg_t:
        return "negative"
    return "neutral"

print(f"{'compound':>9}  {'label':<9}  text")
print("-" * 70)
for t in examples:
    c = vader_scores(t)["compound"]
    print(f"{c:>9.3f}  {vader_label(t):<9}  {t[:48]}")

**Read the output.** The clearly positive and negative sentences land where you expect. *"This is NOT a good product"* should come out negative — that is the negation rule earning its keep; a plain word count would call it positive because it contains *"good."*

Notice the **failures**, too. The sarcastic *"Great. Another broken update."* will likely read as *negative* (because of "broken") or even *positive* (because of "Great") — lexicons have **no model of sarcasm**. And the mixed *"gorgeous but ... outrageous"* collapses two opposite opinions into one number. Hold those two cases in mind; they motivate the next two sections.

> **When to use a lexicon.** No labels, no GPU, need an answer *now*, text is short and informal, and "roughly right, fully transparent" beats "more accurate but a black box." Social-media monitoring dashboards and quick triage are the classic fits.

## 2. Classical ML — TF-IDF + Logistic Regression  ⭐ the workhorse

This is the rung you should reach for **most of the time** once you have labels. It is fast to train (seconds), fast to predict (microseconds), runs on a laptop CPU, and — crucially — it is **interpretable**: you can pull out the exact words that push a prediction positive or negative. On in-domain data it is often within a few points of a transformer for a tiny fraction of the cost.

The pipeline is two steps:

1. **`TfidfVectorizer`** turns text into a numeric matrix. *TF-IDF* = term frequency × inverse document frequency: a word counts for more if it is frequent *in this document* but rare *across all documents*. Common words like *the* get down-weighted automatically.
2. **`LogisticRegression`** learns a weight per word. Sum the weights of the words present, push through a sigmoid → probability of "positive."

Let's build it on a small inline labeled dataset of product/support reviews (`1` = positive, `0` = negative).

In [ ]:
# Small inline labeled dataset: short product / support reviews.
# label 1 = positive, 0 = negative. We deliberately re-use a core sentiment
# vocabulary (love/great/excellent vs terrible/broke/slow) ACROSS reviews so
# held-out reviews share words with the training set -- that overlap is exactly
# what lets a bag-of-words model generalise. The last few rows of each class
# are "hard" cases that mix in the other side's words (e.g. "Looks great but
# broke...") to keep the task honestly imperfect.
reviews = [
    ("Absolutely love this product, it works perfectly and looks great", 1),
    ("Best purchase this year, fast delivery and excellent quality", 1),
    ("Great value for money, works perfectly and easy to use", 1),
    ("Excellent build quality and the battery life is great", 1),
    ("Super happy with it, customer support was fast and friendly", 1),
    ("Easy to set up and it just works, highly recommend", 1),
    ("Fantastic quality and great value, would buy again", 1),
    ("Love the sleek design, fast and incredibly easy to use", 1),
    ("Works perfectly, great quality and arrived early, recommend it", 1),
    ("Reliable and well made, excellent value and easy to use", 1),
    ("My favourite gadget, fast reliable and flawless so far", 1),
    ("Great app, easy to use and works perfectly with no bugs", 1),
    ("Amazing quality and the battery lasts forever, love it", 1),
    ("Highly recommend, excellent product and fast friendly support", 1),
    ("Perfect for the price, works great and easy to set up", 1),
    ("Really happy with the quality, fast and reliable, love it", 1),
    ("Great product, excellent value and works perfectly", 1),
    ("Fast, reliable and easy to use, best purchase I have made", 1),
    ("Love it, great design and excellent build quality", 1),
    ("Works great, easy to use and fantastic value for money", 1),
    ("Excellent customer support, fast friendly and very helpful", 1),
    ("Great quality product, reliable and easy to recommend", 1),
    ("Perfectly happy, works great and the quality is excellent", 1),
    ("Fast delivery, great packaging and the product works perfectly", 1),
    ("Reliable and fast, love the quality and easy to use", 1),
    ("Best value, excellent quality and works perfectly every time", 1),
    ("Was worried it would be slow but it works perfectly, love it", 1),  # hard
    ("Not the cheapest but excellent quality and fast, worth it", 1),     # hard
    ("Had a slow start but support fixed it fast, very happy now", 1),    # hard
    ("A little pricey but reliable and well made, recommend it", 1),      # hard
    ("Terrible quality, it broke after two days, complete waste of money", 0),
    ("Worst purchase ever, slow and constantly crashes", 0),
    ("Awful product, broke immediately and customer support was useless", 0),
    ("Cheaply made and stopped working, very disappointed", 0),
    ("Do not buy this, it is slow and the quality is terrible", 0),
    ("Broke on day one, poor quality and a complete waste of money", 0),
    ("Disappointed with the quality, slow and crashes constantly", 0),
    ("Terrible customer support, the product is defective and slow", 0),
    ("Worst quality, it broke quickly and was a waste of money", 0),
    ("Poor build quality, the screen cracked and it stopped working", 0),
    ("Awful experience, arrived damaged and the quality is terrible", 0),
    ("Slow and buggy, crashes constantly, very disappointed", 0),
    ("Hate it, defective on arrival and customer support was useless", 0),
    ("Complete waste of money, broke fast and poor quality", 0),
    ("Terrible value, slow and stopped working after a week", 0),
    ("Defective product, broke immediately, requesting a refund", 0),
    ("Poor quality and slow, very disappointed, would not recommend", 0),
    ("Worst product ever, crashes constantly and feels cheaply made", 0),
    ("Awful quality, broke quickly and the support was terrible", 0),
    ("Slow, buggy and unreliable, a complete waste of money", 0),
    ("Disappointed, poor quality and it stopped working fast", 0),
    ("Terrible and defective, broke on day one, waste of money", 0),
    ("Cheaply made, slow and crashes, hate this product", 0),
    ("Useless support, defective product and terrible quality", 0),
    ("Broke after a week, poor quality and very disappointed", 0),
    ("Awful and slow, constantly crashes, worst purchase ever", 0),
    ("Looks great but broke after two days, very disappointed", 0),  # hard
    ("Loved it at first but it stopped working, poor quality", 0),    # hard
    ("Fast delivery but the product is defective and useless", 0),    # hard
    ("Great design but slow, buggy and a waste of money", 0),         # hard
]
texts  = [t for t, _ in reviews]
labels = np.array([y for _, y in reviews])
print(f"{len(texts)} reviews  |  positives: {labels.sum()}  "
      f"negatives: {(labels == 0).sum()}")

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.30, random_state=1, stratify=labels)

clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words="english")),
    ("lr", LogisticRegression(max_iter=1000)),
])
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"Test accuracy: {acc:.2%}  (on {len(y_test)} held-out reviews)")
print()
# zero_division=0: if a class is never predicted on this tiny test set, report
# 0 for its precision instead of raising an UndefinedMetricWarning.
print(classification_report(y_test, pred,
                            target_names=["negative", "positive"],
                            zero_division=0))

# A single 18-row split is still noisy; 5-fold CV on all 60 rows is a steadier
# read. Both land high here because the reviews share vocabulary in-domain.
cv = cross_val_score(clf, texts, labels, cv=5)
print(f"5-fold CV accuracy: {cv.mean():.2%} +/- {cv.std():.2%}")
print()
print("Note: this is still a tiny toy dataset -- real projects use thousands\n"
      "of labels. The point is the workflow + diagnostics (confusion matrix,\n"
      "per-class report, CV), and that an in-domain bag-of-words baseline\n"
      "comfortably beats the generic lexicon from Section 1.")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["neg", "pos"], yticklabels=["neg", "pos"], ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title(f"Confusion matrix (acc = {acc:.0%})")
plt.tight_layout(); plt.show()

### Inspecting `.coef_` — *which words* drive the model

This is the superpower of the linear model. `LogisticRegression` learns one weight per TF-IDF feature. A large **positive** weight means "this token pushes toward the positive class"; a large **negative** weight pushes toward negative. Mapping weights back to the vocabulary gives you a human-readable explanation of the model — invaluable for **debugging** ("why did it flag this ticket?") and for **trust** with stakeholders.

In [ ]:
vec = clf.named_steps["tfidf"]
lr = clf.named_steps["lr"]
vocab = np.array(vec.get_feature_names_out())
coefs = lr.coef_[0]

order = np.argsort(coefs)
top_neg = order[:10]
top_pos = order[-10:][::-1]

print("Most POSITIVE tokens:")
for i in top_pos:
    print(f"  {coefs[i]:+.3f}  {vocab[i]}")
print("\nMost NEGATIVE tokens:")
for i in top_neg:
    print(f"  {coefs[i]:+.3f}  {vocab[i]}")

In [ ]:
# Visualise the most influential tokens
sel = np.concatenate([top_pos[::-1], top_neg[::-1]])
vals = coefs[sel]
colors = ["#2a9d8f" if v > 0 else "#e76f51" for v in vals]
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.barh(range(len(sel)), vals, color=colors)
ax.set_yticks(range(len(sel)))
ax.set_yticklabels(vocab[sel])
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("logistic-regression weight")
ax.set_title("Tokens that drive the prediction")
plt.tight_layout(); plt.show()

In [ ]:
# Use the trained classifier on our shared examples + probabilities
proba = clf.predict_proba(examples)[:, 1]
print(f"{'P(positive)':>11}  {'label':<9}  text")
print("-" * 70)
for t, p in zip(examples, proba):
    lab = "positive" if p >= 0.5 else "negative"
    print(f"{p:>11.2f}  {lab:<9}  {t[:48]}")

Compare these probabilities to VADER's labels from Section 1. The classical model learned vocabulary *from our review domain*, so words like *outrageous* or *broken* carry the weight our data taught them — whereas VADER uses a general-purpose dictionary. **This domain-fit is exactly why a trained baseline usually beats a generic lexicon on your own data.**

> **When to use classical ML.** You have (or can label) at least a few hundred examples, you need it fast and cheap and explainable, and the text resembles your training data. This is the default professional choice — build it *first*, then only escalate if it isn't good enough.

## 3. Transformers — Hugging Face `pipeline`

Transformer models (BERT, DistilBERT, RoBERTa…) are pretrained on enormous text corpora and then **fine-tuned** for sentiment. Because they model words *in context* (via attention), they handle negation, long-range dependencies, and subtle phrasing far better than a bag-of-words model — at the cost of size (hundreds of MB) and latency (milliseconds–seconds, ideally on a GPU).

Hugging Face's `pipeline` hides all the tokenizer/model plumbing:

```python
from transformers import pipeline
clf = pipeline('sentiment-analysis')        # downloads a default model
clf(['I love this', 'terrible'])
# -> [{'label': 'POSITIVE', 'score': 0.9998},
#     {'label': 'NEGATIVE', 'score': 0.9995}]
```

With no model name, the default is a **`distilbert-base-uncased`** model fine-tuned on **SST-2** (the Stanford Sentiment Treebank) — a binary POSITIVE/NEGATIVE classifier. The returned `score` is the model's confidence in the chosen `label`.

`transformers` is large and downloads a model on first use, so it often isn't available offline. When it's missing we **gracefully fall back** to the classical model trained in Section 2 — same interface, returns `{label, score}` — so the rest of the notebook still runs. (A trained baseline is, in fact, a perfectly reasonable production fallback when a transformer is unavailable.)

### Domain matters — pick the right pretrained model

The default SST-2 model was trained on *movie reviews*. For other domains, use a model fine-tuned on matching text:

| Model | Domain | Labels |
|---|---|---|
| `distilbert-...-sst-2` (default) | general / reviews | POS / NEG |
| **FinBERT** (`ProsusAI/finbert`) | financial news, filings | pos / neg / neutral |
| **twitter-roberta-base-sentiment** | tweets, social | neg / neu / pos |
| `nlptown/bert-base-multilingual-uncased-sentiment` | multilingual reviews | 1–5 stars |

Using a finance model on tweets (or vice-versa) is a common cause of bad results — **domain shift**, which we revisit in the pitfalls section.

In [ ]:
# Stand-in: reuse the Section-2 classical model, mimic the HF output shape.
def _classical_sentiment(texts):
    p = clf.predict_proba(list(texts))[:, 1]
    out = []
    for prob in p:
        if prob >= 0.5:
            out.append({"label": "POSITIVE", "score": float(prob)})
        else:
            out.append({"label": "NEGATIVE", "score": float(1 - prob)})
    return out

USING_REAL_TRANSFORMER = False
if HAS_TRANSFORMERS:
    # transformers can import yet still fail here if the model weights are not
    # cached and there is no network -- so guard the download too, not just the
    # import, and fall back to the classical model either way.
    try:
        print("Loading real Hugging Face pipeline "
              "(first run downloads the model)...")
        hf = pipeline("sentiment-analysis")
        def sentiment_transformer(texts):
            return hf(list(texts))
        USING_REAL_TRANSFORMER = True
        print("Using REAL transformers pipeline.")
    except Exception as e:
        sentiment_transformer = _classical_sentiment
        print(f"Could not load a transformer model ({type(e).__name__}) -> "
              "falling back to the STAND-IN (classical model from Section 2).")
else:
    sentiment_transformer = _classical_sentiment
    print("transformers not installed -> using STAND-IN "
          "(the classical model from Section 2).")

In [ ]:
results = sentiment_transformer(examples)
print(f"{'label':<9} {'score':>6}   text")
print("-" * 70)
for t, r in zip(examples, results):
    print(f"{r['label']:<9} {r['score']:>6.3f}   {t[:48]}")

With the **real** pipeline, look closely at the sarcastic and negated examples — context-aware models usually get *"This is NOT a good product"* right with high confidence, and handle subtle phrasing the lexicon missed. They still struggle with genuine sarcasm (no model fully solves it) and they are far heavier to run.

> **When to use a transformer.** Accuracy on hard, context-heavy text is worth the cost; you have GPU/latency budget or batch offline; or you can use a strong **domain-specific** pretrained model and skip training entirely. For real-time, high-volume, latency-sensitive paths, a distilled model or the classical baseline is often the smarter engineering call.

## 4. Side-by-side: the three rungs of the ladder

The same six sentences, scored by all three approaches. This is the table to internalise — not as "transformer wins," but as *"each method has a job."*

In [ ]:
hf_res = sentiment_transformer(examples)
print(f"{'VADER':<10}{'Classical':<12}{'Transf.':<10} text")
print("-" * 78)
for t, hr, p in zip(examples, hf_res, clf.predict_proba(examples)[:, 1]):
    v = vader_label(t)
    c = "positive" if p >= 0.5 else "negative"
    h = hr["label"].lower()
    print(f"{v:<10}{c:<12}{h:<10} {t[:42]}")

if not USING_REAL_TRANSFORMER:
    print("\n(No real transformer here, so the Transf. column reuses the "
          "classical model -> it matches the Classical column by design. "
          "Install transformers + torch to see them diverge.)")

In [ ]:
# A cheat-sheet you can keep.
import textwrap
summary = """
               LEXICON (VADER)     CLASSICAL ML         TRANSFORMER
  needs labels?   no                yes (100s+)          no (use pretrained)
  training        none              seconds              hours (or none)
  inference       instant           microseconds         ms-seconds
  hardware        CPU               CPU                  GPU preferred
  interpretable   yes (rules)       yes (.coef_)         hard
  context/negation weak-medium      weak (bag-of-words)  strong
  best for        short social txt  in-domain workhorse  hard context text
"""
print(summary)

## 5. Aspect-based sentiment (ABSA) — *what* are they happy/unhappy about?

A single label per review throws away information. Consider:

> *"The **screen** is gorgeous but the **battery** is a disaster and the **price** is too high."*

Overall sentiment ≈ negative — but that hides the real insight. **Aspect-based sentiment analysis** assigns a sentiment to each *aspect* (feature/topic) mentioned: `screen → positive`, `battery → negative`, `price → negative`. This is what turns a sentiment dashboard from a mood-ring into a **product roadmap**: you can see *exactly which feature* is dragging satisfaction down.

Full ABSA is its own modeling problem (aspect *extraction* + per-aspect *classification*, often with dedicated transformer models). But a surprising amount of business value comes from a simple **keyword + local-window** heuristic: for each aspect keyword, score the surrounding text. Below is a toy version to make the idea concrete.

In [ ]:
aspects = {
    "screen":   ["screen", "display"],
    "battery":  ["battery", "charge"],
    "price":    ["price", "expensive", "cheap", "value", "cost"],
    "support":  ["support", "service", "staff"],
}

def aspect_sentiment(text, window=4):
    """Toy ABSA: score a small word-window around each aspect keyword.
    The window must be tight -- too wide and an aspect catches the sentiment
    of a neighbouring clause (e.g. 'battery' soaking up 'gorgeous' from the
    screen clause). That fragility is the whole point: it is why production
    ABSA uses clause segmentation or a dedicated model, not a fixed window."""
    words = re.findall(r"[A-Za-z']+", text.lower())
    found = {}
    for aspect, keys in aspects.items():
        idxs = [i for i, w in enumerate(words) if w in keys]
        if not idxs:
            continue
        chunks = []
        for i in idxs:
            lo, hi = max(0, i - window), min(len(words), i + window + 1)
            chunks.append(" ".join(words[lo:hi]))
        c = vader_scores(" ".join(chunks))["compound"]
        found[aspect] = ("positive" if c >= 0.05 else
                         "negative" if c <= -0.05 else "neutral")
    return found

review = ("The screen is gorgeous and bright, but the battery is a disaster "
          "and the price is outrageous. Support was helpful though.")
print("Review:", review, "\n")
for aspect, sent in aspect_sentiment(review).items():
    print(f"  {aspect:<8} -> {sent}")

That heuristic is crude (a fixed word-window, lexicon scoring) but it demonstrates the shape of the answer ABSA gives you. In production you would reach for a dedicated ABSA model or an LLM with a structured prompt, but **aggregating per-aspect sentiment across thousands of reviews** is one of the highest-ROI things a data team can ship for a product org.

## 6. Pitfalls & practitioner notes

Sentiment analysis looks easy in a demo and bites you in production. The failures are rarely about the model architecture — they are about the **data, the labels, and the framing**. Internalise these.

**1. Sarcasm & irony.** *"Great, another broken update."* Lexicons and bag-of-words models get this wrong essentially always; even transformers struggle without context. There is no clean fix — flag it as a known error mode and don't over-trust positive scores on complaint channels.

**2. Domain shift.** A model trained on movie reviews (or finance news) will degrade on tweets, support tickets, or medical text. *"This drug is insane"* is positive in slang, alarming in a clinical note. **Always validate on text from your actual domain**, and prefer a domain-matched pretrained model (FinBERT for finance, twitter-roberta for social).

**3. The neutral class is real — and hard.** Binary POS/NEG models *force* every input into a side, so factual or mixed text (*"It arrived Tuesday"*) gets a confident-but-meaningless label. If neutral matters to your business, model it explicitly (3-class) or use a confidence threshold to abstain.

**4. Class imbalance.** Real feedback is often 90% positive (or, on a complaints line, 90% negative). A model that always predicts the majority can hit 90% accuracy and be useless. Look at **per-class precision/recall** (we used `classification_report` above), not just accuracy, and consider class weights or resampling.

**5. Calibration.** A `score` of 0.92 is *not* a 92% probability of being correct unless the model is calibrated. Don't threshold on raw transformer scores for high-stakes routing without checking calibration (e.g. a reliability curve) or using `CalibratedClassifierCV` for the classical model.

**6. Mapping star ratings → sentiment.** Tempting to label *4–5★ = positive, 1–2★ = negative, 3★ = neutral* and call it free labels. It mostly works, but **the text and the stars often disagree** (a glowing review with a mis-clicked 1★; a 5★ "it's fine"). Treat star-derived labels as *noisy*, and spot-check. Also remember cultural rating bias — some user bases rarely give 5★, others rarely give 1★.

## 🧪 Exercises

These are open-ended — there is no autograder in the optional modules. Edit the cells above and observe.

1. **Threshold tuning.** VADER's neutral band is `[-0.05, 0.05]`. Widen it to `[-0.3, 0.3]` in `vader_label`. How many of the example texts become neutral? When would a wider band help a business (hint: triage volume)?

2. **Break the classical model.** Add 4–6 of your *own* reviews (mix of positive/negative, include a sarcastic one) to the `reviews` list, re-run, and look at the new `.coef_` top tokens. Did your words show up?

3. **n-grams matter.** Change `ngram_range=(1, 2)` to `(1, 1)` in the `TfidfVectorizer`. Does *"not good"* still get handled? Explain why bigrams help a bag-of-words model with negation.

4. **Add a neutral class.** Add a few genuinely neutral reviews labeled `2` and retrain a 3-class model. What happens to accuracy and to the confusion matrix?

5. **(If you install transformers)** Swap the default pipeline for `pipeline('sentiment-analysis', model='cardiffnlp/twitter-roberta-base-sentiment-latest')` and compare on a few tweets. Does the social-domain model handle slang/emoji better?

6. **Extend ABSA.** Add an `'shipping'` aspect (keywords: delivery, arrived, shipping, late) and test it on a few of your own reviews.

## 🧠 Key takeaways

- Sentiment analysis is a **ladder**, not a single model. Climb only as high as the problem requires.
- **Lexicon (VADER):** zero training, instant, transparent; engineered rules handle negation, intensifiers, CAPS, punctuation, emoji. Best for short, informal text and quick reads.
- **Classical ML (TF-IDF + LogisticRegression):** the **workhorse baseline** — cheap, fast, interpretable via `.coef_`, and hard to beat in-domain. *Build this first.*
- **Transformers:** strongest on context, negation, and subtlety; heavier and slower. Pick a **domain-matched** pretrained model (FinBERT, twitter-roberta) rather than the generic default.
- **Aspect-based** sentiment (per-feature) is where the real product insight lives.
- The hard part is rarely the model — it's **sarcasm, domain shift, the neutral class, class imbalance, calibration, and noisy star-derived labels.** Always evaluate per-class on your *own* domain's text.
- Guard optional dependencies behind capability flags so your code degrades gracefully — a trained baseline is a fine fallback for an unavailable transformer.

## 🚀 Next step

You have now turned messy text into a business signal at three levels of sophistication. The natural continuation is moving from text to **structured / tabular** data with deep learning:

➡️ **[NB 46 — DeepTab: tabular deep learning](../13_DeepTab/46_deeptab_tabular_deep_learning.ipynb)**

…or head back to the **[Module 12 README](./README.md)** to pick another optional notebook.

---

*Notebook 45 — Sentiment Analysis · Module 12 (Optional) · Python for AI-Driven Automation and Business Data Science.*